In [ ]:
samples = ["A1", "A2", "B2", "C2", "D1"]
zarr_file_path = "/scratch/leuven/357/vsc35768/spatial-transcriptomics/intermediate_results/20260126_final.zarr"
scvi_model_path = "/scratch/leuven/357/vsc35768/spatial-transcriptomics/intermediate_results/model_20260128_final"
outdir = "figures"

In [ ]:
import spatialdata as sd
import scvi
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import os

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns
import spatialdata_plot

## Loading the data

In [ ]:
# loading zarr
sdata = sd.read_zarr(zarr_file_path)
sdata

In [ ]:
# loading the model
scvi_model = scvi.model.SCVI.load(scvi_model_path)
# get the anndata from the model
adata_scvi = scvi_model.adata
adata_scvi

## Differential Expression analysis - scVI

In [ ]:
exclude_genes = [
     "Rbfox3",
     "Tubb3",
     "Snap25",
     "Syt1",
     "Slc17a7",
     "Slc17a6",
     "Gad1",
     "Gad2",
     "Aldoc",
     "Aqp4",
     "Aldh1l1",
     "Slc1a3",
     "Slc1a2",
     "Gfap",
     "Frzb",
     "Ascl1",
     "Agt",
     "Fam107a",
     "Plp1",
     "Mog",
     "Csf1r",
     "C1qa",
     "Flt1",
     "Pecam1",
]

In [ ]:
cell_type_1 = "Astrocytes Protoplasmic"
cell_type_2 = "CA1 Neurons"
cell_type_3 = "CA2/CA3 Neurons"
cell_type_4 = "DG Granular Neurons"

In [ ]:
# subsetting
astro_ca1_ca3_dg = adata_scvi[adata_scvi.obs["cell_type"].isin([cell_type_1, cell_type_2, cell_type_3, cell_type_4])].copy()

### Astrocytes vs CA1 Neurons

In [ ]:
# DEG analysis
de_ca1 = scvi_model.differential_expression(
    idx1 = adata_scvi.obs["cell_type"] == cell_type_1,  
    idx2 = adata_scvi.obs["cell_type"] == cell_type_2,
    weights = "uniform",
    batch_correction = True,
    filter_outlier_cells = True,
    mode = "change",
)
de_ca1

In [ ]:
# filtering the dataframe on values and the marker genes
de_ca1 = de_ca1[(de_ca1["is_de_fdr_0.05"]) & (abs(de_ca1.lfc_mean) > 1)]
de_ca1_filt_genes = de_ca1.loc[~de_ca1.index.isin(exclude_genes)]
#de_ca1_filt_raw_norm = de_ca1_filt_genes[(de_ca1_filt_genes.raw_normalized_mean1 > 3) | (de_ca1_filt_genes.raw_normalized_mean2 > 3)]
de_ca1_filt_genes

In [ ]:
# get the top 3 of the neurons and all the astrocyte genes
top5_neuron = de_ca1_filt_genes.nsmallest(3, "lfc_mean")
astrocyte = de_ca1_filt_genes.loc[de_ca1["lfc_mean"] > 0]

de_keep_ca1 = (
    pd.concat([top5_neuron, astrocyte], axis=0)
      .drop_duplicates()
)
de_keep_ca1 = de_keep_ca1.sort_values("lfc_mean", ascending=False)

de_keep_ca1

In [ ]:
ca1_genes = de_keep_ca1.index.tolist()

### Astrocytes vs CA2/Ca3 Neurons

In [ ]:
# DEG analysis
de_ca3 = scvi_model.differential_expression(
    idx1 = adata_scvi.obs["cell_type"] == cell_type_1,  
    idx2 = adata_scvi.obs["cell_type"] == cell_type_3,
    weights = "uniform",
    batch_correction = True,
    filter_outlier_cells = True,
    mode = "change",
)
de_ca3

In [ ]:
# filtering the dataframe on values and the marker genes
de_ca3 = de_ca3[(de_ca3["is_de_fdr_0.05"]) & (abs(de_ca3.lfc_mean) > 1)]
de_ca3_filt_genes = de_ca3.loc[~de_ca3.index.isin(exclude_genes)]
#de_ca1_filt_raw_norm = de_ca1_filt_genes[(de_ca1_filt_genes.raw_normalized_mean1 > 3) | (de_ca1_filt_genes.raw_normalized_mean2 > 3)]
de_ca3_filt_genes

In [ ]:
# get the top 3 of the neurons and all the astrocyte genes
top5_neuron = de_ca3_filt_genes.nsmallest(3, "lfc_mean")
astrocyte = de_ca3_filt_genes.loc[de_ca3["lfc_mean"] > 0]

de_keep_ca3 = (
    pd.concat([top5_neuron, astrocyte], axis=0)
      .drop_duplicates()
)
de_keep_ca3 = de_keep_ca3.sort_values("lfc_mean", ascending=False)

de_keep_ca3

In [ ]:
ca3_genes = de_keep_ca3.index.tolist()

### Astrocytes vs DG Granule Neurons

In [ ]:
# DEG analysis
de_dg = scvi_model.differential_expression(
    idx1 = adata_scvi.obs["cell_type"] == cell_type_1,  
    idx2 = adata_scvi.obs["cell_type"] == cell_type_4,
    weights = "uniform",
    batch_correction = True,
    filter_outlier_cells = True,
    mode = "change",
)
de_dg

In [ ]:
# filtering the dataframe on values and the marker genes
de_dg = de_dg[(de_dg["is_de_fdr_0.05"]) & (abs(de_dg.lfc_mean) > 1)]
de_dg_filt_genes = de_dg.loc[~de_dg.index.isin(exclude_genes)]
#de_ca1_filt_raw_norm = de_ca1_filt_genes[(de_ca1_filt_genes.raw_normalized_mean1 > 3) | (de_ca1_filt_genes.raw_normalized_mean2 > 3)]
de_dg_filt_genes

In [ ]:
# get the top 3 of the neurons and all the astrocyte genes
top5_neuron = de_dg_filt_genes.nsmallest(3, "lfc_mean")
astrocyte = de_dg_filt_genes.loc[de_dg["lfc_mean"] > 0]

de_keep_dg = (
    pd.concat([top5_neuron, astrocyte], axis=0)
      .drop_duplicates()
)
de_keep_dg = de_keep_dg.sort_values("lfc_mean", ascending=False)

de_keep_dg

In [ ]:
dg_genes = de_keep_ca3.index.tolist()

### Plotting of the DEG analyses

In [ ]:
m = pd.concat(
    [
        de_keep_ca1["lfc_mean"].rename("Astro_vs_CA1"),
        de_keep_ca3["lfc_mean"].rename("Astro_vs_CA23"),
        de_keep_dg["lfc_mean"].rename("Astro_vs_DG"),
    ],
    axis=1
)
m = m.fillna(0.0)
m

In [ ]:
# ranking the genes for the plot
rank = m.sum(axis=1)
m_sorted = m.loc[rank.sort_values(ascending=False).index]

plt.figure(figsize=(3, 5))
sns.heatmap(
    m_sorted,
    center=0,
    cmap="PRGn",
    linewidths=0.2,
    linecolor="white",
)
plt.xlabel("")
plt.ylabel("")
plt.tight_layout()

plt.savefig(
    "figures/astro_vs_neurons_logFC_heatmap.svg",
    format="svg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
genes_to_show = m_sorted.index.tolist()

dp = sc.pl.dotplot(
    astro_ca1_ca3_dg,
    var_names = genes_to_show,
    groupby = "cell_type",
    cmap = "BuPu",
    use_raw = False,
    return_fig = True,
    vmax = 2,
    show = False,
    figsize = (3,4),
    swap_axes=True
)
dp.style(dot_edge_color='black', dot_edge_lw=0).show()

dp.savefig("figures/dp_astro_ca1_ca3_dg.svg", bbox_inches="tight")

## Spatial plots of candidates

In [ ]:
genes_to_plot = [
    "Gpr37l1",
    "Hepacam",
    "Vcam1",
    "Plxnb1",
    "Ephb4",
    "Slc3a2",
    "Cadm4",
    "Neo1",
    "Lsamp",
    "Alcam"
]

In [ ]:
tbl = sdata.tables["A1_transcriptomics_filtered"]
binary_cmap = ListedColormap(["lightgray", "black"])

for gene in genes_to_plot:
    # get binary 
    x = tbl[:, gene].X
    x = x.toarray().ravel()
    tbl.obs[f"{gene}_expressed"] = (x > 0).astype(int)

    # plotting
    sdata.pl.render_labels(
        "A1_segmentation_mask", 
        color = f"{gene}_expressed", 
        method = "datashader", 
        fill_alpha = 0.5, 
        table_name = "A1_transcriptomics_filtered",
        cmap = binary_cmap
    ).pl.show(
        coordinate_systems = "A1",
        dpi=600,
        title = gene
    )

    fig = plt.gcf()
    fn = os.path.join(outdir, f"{gene}_spatial_expression.svg")
    fig.savefig(fn, format="svg", bbox_inches="tight", dpi=600)
    plt.close(fig)
    